# AI Risk Analyzer - Scam Detection Training
This notebook contains the complete pipeline for training both an LSTM and a GRU deep learning model to classify job and internship postings as Safe, Suspicious, or High Risk (Scam).

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, GRU, Dense, Bidirectional, Dropout
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import pickle
import json

## 1. Dataset Handling
We generate an expanded sample dataset to demonstrate the pipeline. This includes diverse patterns like payment-for-equipment, fake fees, and informal interview requests.

In [ ]:
data = {
    'text': [
        # SAFE (0)
        'We are hiring software engineers, please apply on our official portal.',
        'Looking for an intern. Apply with resume at standardcompany.com',
        'Please review the job description attached and submit your portfolio.',
        'Your interview with the engineering team is scheduled for next Tuesday at 10 AM.',
        'Hello, we saw your profile on LinkedIn and would like to discuss a Senior DevOps role.',
        'Welcome to the team! Attached is your official onboarding guide and employee handbook.',
        'Please complete the take-home technical assessment using the link provided in the invite.',
        'The role requires 3 years of experience in React and Node.js. Salary is competitive.',

        # SUSPICIOUS (1)
        'You have been selected. Send details to confirm.',
        'Looking for a data entry operator. High salary for 2 hours of work. Reply fast.',
        'We found your resume on a public database. Contact the hiring manager on Telegram.',
        'Interview conducted via Google Hangouts or WhatsApp only. Please be online.',
        'Work from home guaranteed without any experience required. Apply now.',
        'Our company is expanding rapidly. We need managers across all locations immediately.',
        'Job offer in International Shipping. No interview needed, direct selection.',
        'Are you interested in earning extra income? We have a great opportunity for you.',

        # HIGH RISK / SCAM (2)
        'Earn money fast! Pay a registration fee of 100 dollars to secure your job.',
        'Urgent requirement! Click the unknown link to get registered immediately.',
        'Congratulations! You won the internship. Pay 50 dollars for training material.',
        'We will send you a check to buy office equipment. Deposit it and send back the balance.',
        'To verify your bank account for salary processing, please provide your login credentials.',
        'Secure your position by paying a mandatory government processing fee for the visa.',
        'Earn 5000 dollars weekly by processing crypto payments for our international clients.',
        'You must pay an insurance deposit which is refundable after your first month of work.',
        'Congratulations! You are selected for the job. Pay 200 dollars for background verification.',
        'Immediate hire: Pay for your uniform and ID card processing to start tomorrow.'
    ],
    'label': [0, 0, 0, 0, 0, 0, 0, 0, 
              1, 1, 1, 1, 1, 1, 1, 1, 
              2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
}
df = pd.DataFrame(data)

X = df['text'].values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## 2. Preprocessing Pipeline

In [ ]:
vocab_size = 5000
max_length = 50
trunc_type = 'post'
padding_type = 'post'
oov_tok = '<OOV>'

tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_tok)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_train_pad = pad_sequences(X_train_seq, maxlen=max_length, padding=padding_type, truncating=trunc_type)

X_test_seq = tokenizer.texts_to_sequences(X_test)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_length, padding=padding_type, truncating=trunc_type)

with open('tokenizer.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

## 3. LSTM Model Training

In [ ]:
lstm_model = Sequential([
    Embedding(vocab_size, 64, input_length=max_length),
    Bidirectional(LSTM(64, return_sequences=True)),
    Dropout(0.3),
    Bidirectional(LSTM(32)),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])

lstm_model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
lstm_model.fit(X_train_pad, y_train, epochs=20, validation_data=(X_test_pad, y_test))
lstm_model.save('scam_detector_lstm.h5')

## 4. GRU Model Training

In [ ]:
gru_model = Sequential([
    Embedding(vocab_size, 64, input_length=max_length),
    Bidirectional(GRU(64, return_sequences=True)),
    Dropout(0.3),
    Bidirectional(GRU(32)),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])

gru_model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
gru_model.fit(X_train_pad, y_train, epochs=20, validation_data=(X_test_pad, y_test))
gru_model.save('scam_detector_gru.h5')